<div dir="rtl" align="right">
<b>۱. راه‌اندازی محیط و تنظیمات اولیه</b><br>
برای اجرای این پروژه، به خاطر حجم بالای داده‌ها و پردازش‌های سنگین مدل زبانی، از محیط گوگل کولب و پردازنده گرافیکی (GPU) استفاده کردیم. یه نکته مهم اینه که مقدار seed رو روی ۴۲ ثابت کردیم تا نتایج کارمون قابل تکرار (Reproducible) باشه و هر بار خروجی‌های متفاوتی نگیریم.
</div>

In [3]:
!pip install -q transformers datasets scikit-learn pandas numpy torch

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import set_seed

set_seed(42)

# Verify gpu is working
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware allocated: {device}")

Hardware allocated: cuda


<div dir="rtl" align="right">
<b>۲. ترکیب داده‌ها و مهندسی ویژگی‌ها (Feature Engineering)</b><br>
اینجا سعی کردیم یه تریک خوب بزنیم! به جای اینکه فقط متن ساده نظرات رو به مدل بدیم، متادیتا (مثل برند و اسم محصول) رو با خلاصه و متن اصلی ترکیب کردیم تا مدل کانتکست (Context) خیلی بهتری از محصول داشته باشه. ضمناً برای اینکه تابع Loss توی شبکه عصبی درست کار کنه، لیبل‌های ۱ تا ۵ رو به ۰ تا ۴ شیفت دادیم (موقع خروجی گرفتن دوباره درستش می‌کنیم).
</div>

In [7]:
import pandas as pd

def load_and_engineer_features(train_path, test_path, brand_path):

    df_train = pd.read_csv(train_path, low_memory=False)
    df_test = pd.read_csv(test_path, low_memory=False)
    df_brand = pd.read_csv(brand_path, low_memory=False)

    for df in [df_train, df_test, df_brand]:
        df.columns = df.columns.str.strip()
        df.rename(columns={'asin': 'Asin', 'ASIN': 'Asin'}, inplace=True)

    print(f"Train columns: {list(df_train.columns)}")
    print(f"Brand columns: {list(df_brand.columns)}")

    df_train = df_train.dropna(subset=['overall'])

    df_train = pd.merge(df_train, df_brand, on='Asin', how='left')
    df_test = pd.merge(df_test, df_brand, on='Asin', how='left')

    text_columns = ['brand', 'title', 'summary', 'reviewText']
    for col in text_columns:
        if col in df_train.columns:
            df_train[col] = df_train[col].fillna('')
            df_test[col] = df_test[col].fillna('')

    df_train['combined_text'] = (
        "Brand: " + df_train['brand'] + " | " +
        "Product: " + df_train['title'] + " | " +
        "Summary: " + df_train['summary'] + " | " +
        "Review: " + df_train['reviewText']
    )

    df_test['combined_text'] = (
        "Brand: " + df_test['brand'] + " | " +
        "Product: " + df_test['title'] + " | " +
        "Summary: " + df_test['summary'] + " | " +
        "Review: " + df_test['reviewText']
    )

    df_train['label'] = df_train['overall'].astype(int) - 1

    print(f"Training data shape: {df_train.shape}")
    print(f"Test data shape: {df_test.shape}")

    return df_train, df_test

train_df_full, test_df = load_and_engineer_features(
    'train_data.csv',
    'test_data.csv',
    'title_brand.csv'
)

print("\nSample of the combined text:")
print(train_df_full['combined_text'].iloc[0])

Train columns: ['overall', 'vote', 'verified', 'reviewTime', 'reviewerID', 'Asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime']
Brand columns: ['Asin', 'title', 'brand']
Training data shape: (855679, 15)
Test data shape: (20381, 13)

Sample of the combined text:
Brand: URC | Product: CLIKR-5 Time Warner Cable Remote Control UR5U-8780L | Summary: Cannot Learn | Review: I have an older URC-WR7 remote and thought this would be an upgrade (and because TWC stuck me with it), but this one fails where the other one didn't.  The old remote could go head to head and learn.  I have 2 different Blu Ray players (LG & Panasonic) and this one fails on both.  It cannot learn buttons.  The biggest problem is in streaming when I need to hit the blue, red, yellow, green buttons.  These cannot work either of them, so I have to pull out the remote for that one setting.  I don't know why they give you multiple code methods but no learning.


<div dir="rtl" align="right">
<b>۳. نمونه‌برداری و توکنایز کردن</b><br>
چون دیتاست خیلی بزرگ بود و زمان محدودی داشتیم، یه نمونه ۵۰ هزارتایی به صورت stratify جدا کردیم که نسبت کلاس‌های امتیازی به هم نخوره. برای مدل پایه هم رفتیم سراغ `distilbert-base-uncased` چون هم سبکه، هم دقیقه و برای رم محدود کولب عالیه. طول متن‌ها رو هم روی ۲۵۶ توکن کات کردیم تا به خطای کمبود حافظه (OOM) نخوریم.
</div>

In [8]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding



SAMPLE_SIZE = 50000
if len(train_df_full) > SAMPLE_SIZE:
    _, train_df_sampled = train_test_split(
        train_df_full,
        test_size=SAMPLE_SIZE,
        stratify=train_df_full['label'],
        random_state=42
    )
else:
    train_df_sampled = train_test_split

train_df, val_df = train_test_split(
    train_df_sampled,
    test_size=0.1,
    stratify=train_df_sampled['label'],
    random_state=42
)

print(f"Final Training Set: {len(train_df)} rows")
print(f"Final Validation Set: {len(val_df)} rows")

hf_train = Dataset.from_pandas(train_df[['combined_text', 'label']])
hf_val = Dataset.from_pandas(val_df[['combined_text', 'label']])
hf_test = Dataset.from_pandas(test_df[['combined_text']])

print("\nLoading Tokenizer")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):

    return tokenizer(
        examples["combined_text"],
        truncation=True,
        max_length=256
    )

print("Tokenizing datasets")

encoded_train = hf_train.map(tokenize_function, batched=True)
encoded_val = hf_val.map(tokenize_function, batched=True)
encoded_test = hf_test.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenization complete")

Final Training Set: 45000 rows
Final Validation Set: 5000 rows

Loading Tokenizer
Tokenizing datasets


Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20381 [00:00<?, ? examples/s]

Tokenization complete


<div dir="rtl" align="right">
<b>۴. لود کردن مدل برای Fine-Tuning</b><br>
مدل از پیش‌آموخته رو لود می‌کنیم. هشدارهایی که اینجا پرینت میشه کاملاً طبیعیه؛ مدل داره سر (Head) دسته‌بندی قبلی خودش رو میندازه دور و یه لایه خطی جدید با ۵ تا خروجی (برای ۵ کلاس خودمون) می‌سازه تا از صفر برای این تسک آموزش ببینه.
</div>

In [9]:
from transformers import AutoModelForSequenceClassification

print(f"Fetching the {MODEL_NAME} model")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

model.to(device)

print(f"Model loaded successfully and attached to: {model.device}")

Fetching the distilbert-base-uncased model


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully and attached to: cuda:0


<div dir="rtl" align="right">
<b>۵. پیکربندی آموزش و معیار ارزیابی سفارشی</b><br>
تو صورت پروژه دقیقاً خواسته شده بود که از معیار Micro F1-Score استفاده کنیم، پس یه تابع کاستوم براش نوشتیم و دادیم به Trainer. یه کار مهم دیگه هم استفاده از `fp16=True` بود؛ این کار محاسبات رو با دقت ۱۶ بیتی انجام میده که هم سرعت آموزش رو دو برابر می‌کنه و هم نصف حافظه گرافیکی رو می‌گیره!
</div>

In [15]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import TrainingArguments, Trainer
import torch

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    micro_f1 = f1_score(labels, predictions, average='micro') # Required by project [cite: 82]
    return {"micro_f1": micro_f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # Updated from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=500,
    report_to="none"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    processing_class=tokenizer,   # <--- THE V5 FIX: Changed 'tokenizer' to 'processing_class'
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting the fine-tuning process...")
trainer.train()

print("\nEvaluating the best model on the validation set:")
trainer.evaluate()

Starting the fine-tuning process...


Epoch,Training Loss,Validation Loss,Micro F1
1,0.693932,0.696787,0.720200
2,0.576373,0.671758,0.732600
3,0.494173,0.703210,0.737800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Evaluating the best model on the validation set:


{'eval_loss': 0.7032102346420288,
 'eval_micro_f1': 0.7378,
 'eval_runtime': 9.9979,
 'eval_samples_per_second': 500.104,
 'eval_steps_per_second': 15.703,
 'epoch': 3.0}

<div dir="rtl" align="right">
<b>۶. استنتاج روی دیتای تست و خروجی نهایی</b><br>
در نهایت، با مدل آموزش‌دیده روی داده‌های تست پیش‌بینی رو انجام دادیم. خروجی‌های ۰ تا ۴ رو با یه جمع ساده به همون فرمت استاندارد ۱ تا ۵ ستاره برگردوندیم و فایل `submission.csv` آماده اس.</div>

In [17]:
import pandas as pd
import numpy as np

print("Generating predictions for the test dataset ")
test_predictions = trainer.predict(encoded_test)

predicted_indices = np.argmax(test_predictions.predictions, axis=-1)

predicted_ratings = predicted_indices + 1

print("Formatting the submission file")
submission_df = pd.DataFrame({
    'predicted': predicted_ratings
})

submission_df.to_csv('submission.csv', index=False)

print("\nPreview of submission.csv:")
print(submission_df.head())

Generating predictions for the test dataset 


Formatting the submission file

Preview of submission.csv:
   predicted
0          1
1          1
2          1
3          1
4          1
